# Работу выполнил студент ДПИ25-1 Карчевский Андрей

# Лабораторная работа №6. Работа со строковыми значениями

В работе используются форматирование строк, регулярные выражения и методы сегментации текста из `nltk`.

In [1]:
import re
import pandas as pd
import numpy as np
import xml.etree.ElementTree as ET

# Для заданий по сегментации текста
import nltk
from nltk.tokenize import RegexpTokenizer, sent_tokenize, word_tokenize
from nltk import pos_tag

# Скачаем необходимые модели (если ещё не скачаны)
nltk.download('punkt', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)


True

## Разминка 1
Выведем содержимое словаря построчно в формате `k = v`, выровняв знак `=` по одной колонке и обернув строковые значения в кавычки.

In [ ]:
obj = {
    "home_page": "https://github.com/pypa/sampleproject",
    "keywords": "sample setuptools development",
    "license": "MIT",
}

max_k = max(len(k) for k in obj.keys())
for k, v in obj.items():
    v_fmt = f'"{v}"' if isinstance(v, str) else str(v)
    print(f"{k:<{max_k}} = {v_fmt}")


## Разминка 2
Составим регулярное выражение для поиска номера группы (возможны варианты с буквенным префиксом и пробелами).

In [ ]:
obj = pd.Series(["Евгения гр.ПМ19-1", "Илья пм 20-4", "Анна 20-3"])
pattern_group = re.compile(r"(?:[А-Яа-я]{2}\s*)?\d{2}-\d", flags=re.IGNORECASE)

obj.apply(lambda s: pattern_group.findall(s))


## Разминка 3
Разобьём формулировку задания 2 на слова.

In [2]:
task2_text = "Написать регулярное выражение,которое позволит найти номера групп студентов."
words = re.findall(r"[\w]+", task2_text, flags=re.UNICODE)
words


['Написать',
 'регулярное',
 'выражение',
 'которое',
 'позволит',
 'найти',
 'номера',
 'групп',
 'студентов']

## 1. Форматирование строк
Загрузим данные о рецептах и выведем 5 случайных записей в виде таблицы фиксированной ширины.

In [3]:
recipes = pd.read_csv("recipes_sample.csv")
sample = recipes.sample(5, random_state=42)[["id", "minutes"]]

id_w = 12
min_w = 9

print(f"|{'id':^{id_w}}|{'minutes':^{min_w}}|")
print(f"|{'-'*id_w}|{'-'*min_w}|")
for rid, mins in sample.itertuples(index=False):
    print(f"|{rid:^{id_w}}|{mins:^{min_w}}|")


|     id     | minutes |
|------------|---------|
|   184910   |   80    |
|   80229    |   145   |
|   246733   |   10    |
|   159180   |   40    |
|   374473   |   45    |


Подготовим функцию для получения шагов рецепта по `id` из XML-файла.

In [4]:
def get_steps_by_id(xml_path: str, recipe_id: int) -> list[str]:
    # Ищем нужный рецепт по id и возвращаем список шагов
    tree = ET.parse(xml_path)
    root = tree.getroot()
    for rec in root.findall("recipe"):
        rid = int(rec.findtext("id"))
        if rid == int(recipe_id):
            steps_el = rec.find("steps")
            return [s.text or "" for s in steps_el.findall("step")]
    return []



## 2. Функция `show_info`
Реализуем функцию, формирующую строку-описание рецепта, и проверим её на тесте, затем применим к рецепту с id `170895`.

In [5]:
def show_info(name: str, steps: list[str], minutes: int, author_id: int) -> str:
    title = str(name).strip().title()
    lines = [f'"{title}"', ""]
    for i, step in enumerate(steps, start=1):
        lines.append(f"{i}. {step}")
    lines.append("----------")
    lines.append(f"Автор: {author_id}")
    lines.append(f"Среднее время приготовления: {minutes} минут")
    return "\n".join(lines)

assert (
    show_info(
        name="george s at the cove black bean soup",
        steps=[
            "clean the leeks and discard the dark green portions",
            "cut the leeks lengthwise then into one-inch pieces",
            "melt the butter in a medium skillet , med",
        ],
        minutes=90,
        author_id=35193,
    )
    == '"George S At The Cove Black Bean Soup"\n\n1. clean the leeks and discard the dark green portions\n2. cut the leeks lengthwise then into one-inch pieces\n3. melt the butter in a medium skillet , med\n----------\nАвтор: 35193\nСреднее время приготовления: 90 минут'
)

xml_path = "steps_sample.xml"
recipe_id = 170895

row = recipes.loc[recipes["id"] == recipe_id].iloc[0]
steps = get_steps_by_id(xml_path, recipe_id)

info = show_info(row["name"], steps, int(row["minutes"]), int(row["contributor_id"]))
print(info)


"Leeks And Parsnips  Sauteed Or Creamed"

1. clean the leeks and discard the dark green portions
2. cut the leeks lengthwise then into one-inch pieces
3. melt the butter in a medium skillet , med
4. heat
5. add the garlic and fry 'til fragrant
6. add leeks and fry until the leeks are tender , about 6-minutes
7. meanwhile , peel and chunk the parsnips into one-inch pieces
8. place in a steaming basket and steam 'til they are as tender as you prefer
9. i like them fork-tender
10. drain parsnips and add to the skillet with the leeks
11. add salt and pepper
12. gently sautee together for 5-minutes
13. at this point you can serve it , or continue on and cream it:
14. in a jar with a screw top , add the half-n-half and arrowroot
15. shake 'til blended
16. turn heat to low under the leeks and parsnips
17. pour in the arrowroot mixture , stirring gently as you pour
18. if too thick , gradually add the water
19. let simmer for a couple of minutes
20. taste to adjust seasoning , probably an addi

## 3. Поиск упоминаний времени в шагах
Найдём в шагах рецепта с id `25082` выражения вида: число + пробел + `hour(s)` или `minute(s)`.

In [6]:
rid = 25082
steps_25082 = get_steps_by_id(xml_path, rid)

time_re = re.compile(r"\b\d+\s+(?:hour|hours|minute|minutes)\b", flags=re.IGNORECASE)

for i, step in enumerate(steps_25082, start=1):
    found = time_re.findall(step)
    if found:
        print(f"Шаг {i}: {found}")


Шаг 6: ['20 minutes']
Шаг 8: ['10 minutes']
Шаг 10: ['2 hours']
Шаг 14: ['10 minutes']
Шаг 17: ['20 minutes', '30 minutes']


## 4. Шаблон `this..., but` в начале строки
Построим регулярное выражение и посчитаем, у скольких рецептов в описании встречается этот шаблон.

In [7]:
this_but_re = r"^this[\w ]*\.\.\.,\s*but"

mask = recipes["description"].fillna("").str.contains(this_but_re, flags=re.IGNORECASE, regex=True)
count = int(mask.sum())
count


0

## 5. Удаление пробелов вокруг дробей
Для рецепта с id `72367` уберём пробелы вокруг символа `/` в дробях вида `a / b`.

In [8]:
rid = 72367
steps_72367 = get_steps_by_id(xml_path, rid)

fixed = [re.sub(r"\s*/\s*", "/", s) for s in steps_72367]
fixed


['mix butter , flour , 1/3 c',
 'sugar and 1-1/4 t',
 'vanilla',
 'press into greased 9" springform pan',
 'mix cream cheese , 1/4 c',
 'sugar , eggs and 1/2 t',
 'vanilla beating until fluffy',
 'pour over dough',
 'combine apples , 1/3 c',
 'sugar and cinnamon',
 'arrange on top of cream cheese mixture and sprinkle with almonds',
 'bake at 350 for 45-55 minutes , or until tester comes out clean']

## 6. Количество уникальных слов в шагах
Разобьём все шаги рецептов на слова с помощью `nltk` и посчитаем количество уникальных слов (только алфавитные, без учёта регистра).

In [9]:
tokenizer = RegexpTokenizer(r"[A-Za-z]+")  # слово = последовательность букв латиницы
unique_words = set()

# Итеративный разбор XML, чтобы не хранить всё дерево в памяти
for event, elem in ET.iterparse(xml_path, events=("end",)):
    if elem.tag == "step":
        text = elem.text or ""
        for tok in tokenizer.tokenize(text):
            unique_words.add(tok.lower())
        elem.clear()

len(unique_words)


15139

## 7. Самые длинные описания по числу предложений
Разобьём описания на предложения при помощи `nltk` и найдём 5 рецептов с максимальным числом предложений.

In [13]:
import nltk

nltk.download("punkt", quiet=True)
try:
    nltk.download("punkt_tab", quiet=True)
except Exception:
    pass

from nltk.tokenize import sent_tokenize

desc = recipes["description"].fillna("")
sent_counts = desc.apply(lambda t: len(sent_tokenize(t)))

top5_idx = sent_counts.sort_values(ascending=False).head(5).index
recipes.loc[top5_idx].assign(n_sentences=sent_counts.loc[top5_idx]).sort_values("n_sentences", ascending=False)


,name,id,minutes,contributor_id,submitted,n_steps,description,n_ingredients,n_sentences
18408,my favorite buttercream icing for decorating,334113,30,681465,2008-10-30,12.0,this wonderful icing is used for icing cakes a...,NaN,76
481,alligator claws avocado fritters with chipot...,287008,45,765354,2008-02-19,NaN,a translucent golden-brown crust allows the gr...,9.0,27
22566,rich barley mushroom soup,328708,60,221776,2008-10-03,NaN,this is one of the best soups i've ever made a...,10.0,24
6779,chocolate tea,205348,6,428824,2007-01-14,NaN,i wrote this because there are an astounding l...,NaN,23
16296,little bunny foo foo cake carrot cake with c...,316000,68,689540,2008-07-27,14.0,the first time i made this cake i grated a mil...,NaN,23


## 8. Вывод частей речи над словами
Реализуем функцию, которая печатает теги частей речи ровно над соответствующими словами, используя выравнивание по центру.

In [15]:
import nltk

nltk.download("punkt", quiet=True)
try:
    nltk.download("punkt_tab", quiet=True)
except Exception:
    pass

nltk.download("averaged_perceptron_tagger", quiet=True)
try:
    nltk.download("averaged_perceptron_tagger_eng", quiet=True)
except Exception:
    pass

from nltk import word_tokenize, pos_tag

def pos_pretty(sentence: str) -> str:
    tokens = word_tokenize(sentence)
    tags = [t for _, t in pos_tag(tokens)]
    widths = [max(len(w), len(t)) + 2 for w, t in zip(tokens, tags)]
    line_tags = "".join(t.center(w) for t, w in zip(tags, widths)).rstrip()
    line_words = "".join(wd.center(w) for wd, w in zip(tokens, widths)).rstrip()
    return line_tags + "\n" + line_words

sent = recipes.loc[recipes["id"] == 241106, "name"].iloc[0]
print(pos_pretty(sent))


    JJ      NNS     IN      NNS     VBP     JJ     CC    JJ    NNS
 eggplant  steaks  with  chickpeas  feta  cheese  and  black  olives
